- torch.distributed.new_group 是 PyTorch 分布式通信中用于创建子通信组的核心 API。它允许你将全局的所有进程（World）划分为若干个更小的子集（Subsets），从而实现更细粒度的通信控制。

以下是关于 dist.new_group 的详细参数介绍、核心规则以及结合代码的实战举例：
## 🛠️ 核心参数与功能
- dist.new_group 的基本调用格式如下：
- group = torch.distributed.new_group(ranks=None, backend=None, timeout=...)
- <b>ranks (list[int])</b>：<font color='red'>一个包含进程全局编号（rank）的列表，指定哪些进程属于这个新组</font>。例如 [0, 1, 2] 表示由 rank 0、1、2 组成一个小组。

- <b>backend (str, 可选)</b>：指定该组使用的通信后端。通常与全局初始化时保持一致（如 'nccl' 用于 GPU，'gloo' 用于 CPU），也支持在同一个程序中混合使用不同的后端。

- <b>返回值</b>：<font color='red'>返回一个不透明的通信组句柄（Group Handle）</font>。在后续调用 all_reduce、broadcast 等集合通信算子时，将其传给 group 参数，通信就只会在这个小组内发生。

## ⚠️ 使用 new_group 的两大铁律
在调用 new_group 时，必须严格遵守以下两条规则，否则会导致程序卡死（Deadlock）或报错：
- 全员参与：集群中的所有进程（<font color='red'>即使它不属于这个新组）都必须执行 new_group 这行代码</font>。<font color='green'>因为底层通信后端（如 NCCL）需要所有进程同步交换通信 ID 来完成建组握手。</font>
- 顺序一致：所有进程创建通信组的顺序必须完全相同。如果 rank 0 先创建组 A 再创建组 B，那么 rank 1、2、3 也必须严格按照这个顺序执行。

## 💡 实战代码举例
### 场景一：创建单个子组（如模型并行组）
- 假设我们有 4 个进程（rank 0-3），现在只想让 rank 0 和 rank 1 组成一个小组进行高频通信，而 rank 2 和 3 不参与。

In [ ]:
import torch
import torch.distributed as dist
import os

# 假设已经通过 init_process_group 完成了全局初始化
rank = int(os.getenv('RANK')) 

# 【全员参与】所有 4 个进程都必须执行这行代码！
# 即使 rank 2 和 3 不在 [0, 1] 里，它们也必须跟着“陪跑”完成建组握手
sub_group = dist.new_group(ranks=[0, 1], backend='nccl')

# 【个人认领】只有 rank 0 和 1 会进入这个分支，保存并使用这个组
if rank in [0, 1]:
    tensor = torch.ones(1).cuda()
    # 通信仅发生在 rank 0 和 1 之间，rank 2 和 3 完全不受影响
    dist.all_reduce(tensor, group=sub_group)
    print(f"Rank {rank} 组内求和结果: {tensor.item()}") # 输出 2.0

### 场景二：创建多个子组（如混合并行架构）
- 在复杂的大模型训练中，通常会同时创建张量并行组（TP）和数据并行组（DP）。

In [ ]:
# 假设 world_size = 4，ranks 为 0, 1, 2, 3
rank = dist.get_rank()

# 定义组的划分
tp_ranks_list = [[0, 1], [2, 3]] # 两个张量并行组
dp_ranks_list = [[0, 2], [1, 3]] # 两个数据并行组

tp_group = None
dp_group = None

# 【顺序一致】所有进程必须严格按照先 TP 后 DP 的顺序循环创建
for ranks in tp_ranks_list:
    g = dist.new_group(ranks)
    if rank in ranks:
        tp_group = g # 认领自己的 TP 组

for ranks in dp_ranks_list:
    g = dist.new_group(ranks)
    if rank in ranks:
        dp_group = g # 认领自己的 DP 组

# 后续训练中，rank 1 就可以分别在 tp_group 里做模型切分通信，
# 在 dp_group 里做梯度同步通信，互不干扰。

### 场景三：作为点对点通信（P2P）的标识符
- 除了集合通信，group 还可以作为 send 和 recv 的通信标识符（Tag）。如果两个 rank 之间需要同时进行多路 P2P 通信，使用不同的 group 可以有效区分不同的通信流，避免数据混淆。

In [ ]:
# 创建专属的 P2P 通信组
p2p_group = dist.new_group([0, 1])

if rank == 0:
    dist.send(tensor, dst=1, group=p2p_group)
elif rank == 1:
    dist.recv(tensor, src=0, group=p2p_group)

总结来说，dist.new_group 是构建高效、复杂分布式训练架构（如 Megatron-LM 的 3D 并行）的基石。只要牢记“全员同步创建，各自按需认领”的原则，就能灵活地驾驭多进程间的通信。